In [162]:
import openpyxl
from openpyxl.styles import PatternFill, Border, Side, Alignment, Font
import os

filename = "rack_layout_top_down.xlsx"
if os.path.exists(filename):
    os.remove(filename)
    print(f"Existing file '{filename}' deleted.")

def generate_visual(distribution, loc, rack_ids):
    
    node_positions = distribution['stable_placement']
    try:
        wb = openpyxl.load_workbook(filename)
    except FileNotFoundError:
        wb = openpyxl.Workbook()
        if 'Sheet' in wb.sheetnames:
            del wb['Sheet']
    
    tab_name = f"Rack_{loc}"
    if tab_name in wb.sheetnames:
        del wb[tab_name]
    ws = wb.create_sheet(title=tab_name)

    # Set column widths
    ws.column_dimensions['A'].width = 25
    ws.column_dimensions['B'].width = 25
    ws.column_dimensions['C'].width = 15
    ws.column_dimensions['D'].width = 15

    # Define styles
    header_font = Font(bold=True)
    border = Border(left=Side(style='thin'), right=Side(style='thin'), 
                   top=Side(style='thin'), bottom=Side(style='thin'))
    center_aligned = Alignment(horizontal='center', vertical='center')
    total_font = Font(bold=True)
    total_fill = PatternFill(start_color='FFFF00', end_color='FFFF00', fill_type='solid')
    
    from colorsys import hls_to_rgb
    import hashlib
    
    class AutoColorMap(dict):
        def __missing__(self, key):
            MAX_ATTEMPTS = 10  # Prevent infinite loops
            for _ in range(MAX_ATTEMPTS):
                # Generate color by hashing the key with iteration count
                hash_val = int(hashlib.md5(f"{key}{len(self)}".encode()).hexdigest()[:6], 16)
                hue = hash_val % 360  # Full hue range
                saturation = 50 + hash_val % 45  # 50-95% (vibrant but not neon)
                lightness = 20 + hash_val % 60   # 20-80% (avoid extremes)

                # Convert HSL to RGB
                r, g, b = hls_to_rgb(hue/360, lightness/100, saturation/100)
                color = f"{int(r*255):02X}{int(g*255):02X}{int(b*255):02X}"

                # Calculate color brightness
                brightness = (0.299 * r + 0.587 * g + 0.114 * b)

                # Reject colors that are too white or too dark
                if 0.15 <= brightness <= 0.85:  # Good visibility range
                    self[key] = color
                    return color

            # Fallback to a medium blue if max attempts reached
            self[key] = "4B8DF0"
            return "4B8DF0"
    
    color_map = {
        'compute_nodes': 'FF9999',
        'GPU_nodes': '99CCFF',
        'LAN': '99FF99',
        'NVMe': "4B8DF0"
    }
    
    color_this = AutoColorMap(color_map)
    for color in colors_info:
        color_map[color] = color_this[color]
    for color in LANs:
        color_map[color] = color_this[color]
    # Write headers
    headers = ["Rack Position", "Device", "Wattage (W)", "Weight (kg)"]
    for col, header in enumerate(headers, start=1):
        cell = ws.cell(row=1, column=col, value=header)
        cell.font = header_font
        cell.border = border
        cell.alignment = center_aligned

    # Track occupied positions
    occupied_info = {}
    total_wattage = 0
    total_weight = 0
    occupied_positions = {}
    occupied_label = {}

    # First pass: Identify all positions and their heights
    for device, posy in node_positions.items():
        if 'LAN_' in device:
            lan_parts = '_'.join(device.split('_')[:-1]).split('__')
            lan_type = lan_parts[0]
            switch_model = lan_parts[1]
            for switch in LANs[lan_type]['switch']:
                if switch['model'] == switch_model:
                    height = switch['height']
                    wattage = switch['wattage']
                    weight = switch['weight']
                    break
            
            pos = posy + height - 1
            occupied_label[pos] = f"{pos-height+1}-{pos}"
            for i in range(pos-height+1, pos+1):
                occupied_positions[i] = device
            
            occupied_info[pos] = {
                'device': device,
                'type': 'LAN Switch',
                'height': height,
                'color': color_map.get('LAN', 'FFFFFF'),
                'wattage': wattage,
                'weight': weight
            }
            total_wattage += wattage
            total_weight += weight
        else:
            node_type = '_'.join(device.split('_')[:-1])
            height = colors_info[node_type]['height']
            wattage = colors_info[node_type]['wattage']
            weight = colors_info[node_type]['weight']
            pos = posy + height - 1
            occupied_label[pos] = f"{pos-height+1}-{pos}"
            for i in range(pos-height+1, pos+1):
                occupied_positions[i] = device
            
            occupied_info[pos] = {
                'device': device,
                'type': node_type.replace('_', ' ').title(),
                'height': height,
                'color': color_map.get(node_type, 'FFFFFF'),
                'wattage': wattage,
                'weight': weight
            }
            total_wattage += wattage
            total_weight += weight

    # Second pass: Write data to worksheet
    current_row = 2
    for rack_pos in range(42, 0,-1):
        # Skip if this position is covered by a device that starts above
        if rack_pos in occupied_positions and rack_pos not in occupied_info:
            continue
            
        # Get position label
        position_label = occupied_label.get(rack_pos, str(rack_pos))
        
        # Write rack position
        ws.cell(row=current_row, column=1, value=position_label)
        ws.cell(row=current_row, column=1).border = border
        ws.cell(row=current_row, column=1).alignment = center_aligned

        if rack_pos in occupied_info:
            device_info = occupied_info[rack_pos]
            height = device_info['height']
            
            # Write device info FIRST
            ws.cell(row=current_row, column=2, value=device_info['device'])
            ws.cell(row=current_row, column=3, value=device_info['wattage'])
            ws.cell(row=current_row, column=4, value=device_info['weight'])
            
            # Apply styling to all cells that will be merged
            fill = PatternFill(start_color=device_info['color'], end_color=device_info['color'], fill_type='solid')
            for col in range(1, 5):
                ws.cell(row=current_row, column=col).fill = fill
                ws.cell(row=current_row, column=col).border = border
                ws.cell(row=current_row, column=col).alignment = center_aligned
            
            # Merge cells if height > 1
            if height > 1:
                for col in range(1, 5):
                    ws.merge_cells(
                        start_row=current_row,
                        end_row=current_row + height - 1,
                        start_column=col,
                        end_column=col
                    )
                
                # Apply border to all merged cells
                for row in range(current_row, current_row + height):
                    for col in range(1, 5):
                        ws.cell(row=row, column=col).border = border
            
            current_row += height
        else:
            # Empty position
            for col in range(2, 5):
                ws.cell(row=current_row, column=col, value="")
                ws.cell(row=current_row, column=col).border = border
                ws.cell(row=current_row, column=col).alignment = center_aligned
            
            current_row += 1

    # Add total row
    total_row = current_row + 1
    ws.cell(row=total_row, column=1, value="TOTAL").font = total_font
    ws.cell(row=total_row, column=2, value="").font = total_font
    ws.cell(row=total_row, column=3, value=total_wattage).font = total_font
    ws.cell(row=total_row, column=4, value=total_weight).font = total_font
    
    for col in range(1, 5):
        ws.cell(row=total_row, column=col).border = border
        ws.cell(row=total_row, column=col).fill = total_fill
        ws.cell(row=total_row, column=col).alignment = center_aligned

    # Save the workbook
    wb.save(filename)
    print(f"Excel file '{filename}' has been created with positions from 42 at top to 1 at bottom.")

Existing file 'rack_layout_top_down.xlsx' deleted.


In [163]:
def iterate_on_racks():
    total_wattage = 0
    total_height = 0
    total_count = 0
    filled_boxes = []
    # Create a dictionary to store unique rack configurations
    unique_racks = {}
    rack_summary = []
    for i, info in enumerate(distributions_info):
        if 'rack_config' in distributions_info[info]:
            # Create a hashable representation of the rack configuration
            rack_config = distributions_info[info]['rack_config']
            # Convert the rack_config dictionary to a tuple of sorted items for hashability
            rack_tuple = tuple(sorted(rack_config.items()))

            # Store the rack details with its ID
            rack_details = {
                'rack_id': i+1,
                'rack_config': rack_config,
                'stable_placement': distributions_info[info].get('stable_placement', []),
                'bottom_place_periority': distributions_info[info].get('bottom_place_periority', []),
                'lan_info': distributions_info[info].get('lan_info', {}),
                'device_info': distributions_info[info].get('device_info', {})
            }

            # If we've seen this configuration before, append to the list
            if rack_tuple in unique_racks:
                unique_racks[rack_tuple]['rack_ids'].append(i+1) # append D
                unique_racks[rack_tuple]['count'] += 1
            else:
                # First time seeing this configuration
                unique_racks[rack_tuple] = {
                    'rack_ids': [i+1],
                    'count': 1,
                    'details': rack_details
                }

        # Track the total count regardless of whether we're processing racks
        if 'stable_placement' in distributions_info[info]:
            total_count += len(distributions_info[info]['stable_placement'])
    
    counter = 1
    for _, item in unique_racks.items():
            generate_visual(item['details'],counter,item['rack_ids'])
            counter += 1
    return

In [164]:
import pickle
from collections import Counter
from math import floor
def global_main():
    global colors_info, LANs, Cables, max_box_wattage, max_box_height,  rack_height_u , rack_weight_kg, rack_width_mm, \
            rack_depth_mm, u_height_mm, rack_height_mm, rack_height_u, rack_cg_height_mm, unit_to_cm, u_height_mm, \
            device_to_rackside, racktop_to_ceiling, rack_to_rack, Rack_rows, Rack_rows, distributions_info
    


    with open('Latest_Racks_info.pkl', 'rb') as f:
        colors_info, LANs, Cables, Rack_rows, distributions_info = pickle.load(f)
    
    
    max_box_wattage = 16000
    max_box_height = rack_height_u = 42
    rack_weight_kg = 114.55
    rack_width_mm = 750
    rack_depth_mm = 1200
    u_height_mm = 44.45
    rack_height_mm = rack_height_u * u_height_mm
    rack_cg_height_mm = rack_height_mm / 2
    global_rack_signature = dict()
    unit_to_cm = u_height_mm /10
    # the following are in units
    device_to_rackside = 6.75
    racktop_to_ceiling = 4.5 # assumed 20cm
    rack_to_rack = 4.5 # assumed from side to the adjacent side no spacing
    # adding to the Cables the maximum stretch
    for switch in Cables:
        for cable in Cables[switch]:
            cable['in_rack_stretch'] = floor((cable['length']*100/unit_to_cm) - (2* device_to_rackside))
            
            
if __name__ == "__main__":
    global_main()
    iterate_on_racks()

Excel file 'rack_layout_top_down.xlsx' has been created with positions from 42 at top to 1 at bottom.
Excel file 'rack_layout_top_down.xlsx' has been created with positions from 42 at top to 1 at bottom.
Excel file 'rack_layout_top_down.xlsx' has been created with positions from 42 at top to 1 at bottom.
Excel file 'rack_layout_top_down.xlsx' has been created with positions from 42 at top to 1 at bottom.
Excel file 'rack_layout_top_down.xlsx' has been created with positions from 42 at top to 1 at bottom.
Excel file 'rack_layout_top_down.xlsx' has been created with positions from 42 at top to 1 at bottom.
Excel file 'rack_layout_top_down.xlsx' has been created with positions from 42 at top to 1 at bottom.


In [165]:
distributions_info[0]['lan_info']

{'LAN_2__Z_9xxx_41': {'model': 'Z_9xxx',
  'ports': 64,
  'speed': 400,
  'height': 2,
  'wattage': 1304,
  'uplink_count': 4,
  'uplink_speed': 800,
  'weight': 20,
  'type': 'LAN_2__Z_9xxx',
  'position': 41,
  'id': '41',
  'half_for_split': 16,
  'full_no_split': 19,
  'cables': {'3m_400_copper_eth': 19.0,
   '3m_400s_copper_eth': 8.0,
   '1.5m_400s_copper_eth': 11},
  'minimum_cable_total_length': 71.5645,
  'actual_cable_total_length': 97.5},
 'LAN_3__S_xx64_39': {'model': 'S_xx64',
  'ports': 64,
  'speed': 25,
  'height': 2,
  'wattage': 300,
  'uplink_count': 4,
  'uplink_speed': 100,
  'weight': 12,
  'type': 'LAN_3__S_xx64',
  'position': 39,
  'id': '39',
  'half_for_split': 0,
  'full_no_split': 35,
  'cables': {'3m_25_copper': 22.0, '1.5m_25_copper': 13.0},
  'minimum_cable_total_length': 64.69697500000001,
  'actual_cable_total_length': 85.5}}

In [166]:
distributions_info[1]['device_info']

{'GPU_nodes_21': {'position': 21,
  'id': '21',
  'LAN_2__Z_9xxx_41': {'cable_count': 5,
   'speed': 200,
   'to_external_rack': 0,
   'half_for_split': 1,
   'cables_count': 2.5,
   'cable_type': {'model': '3m_400s_copper_eth',
    'server_port_speed': 200,
    'split': 2,
    'length': 3,
    'in_rack_stretch': 53}},
  'LAN_3__S_xx64_39': {'cable_count': 1,
   'speed': 25,
   'to_external_rack': 0,
   'full_no_split': 1,
   'cables_count': 1.0,
   'cable_type': {'model': '3m_25_copper',
    'server_port_speed': 25,
    'split': 1,
    'length': 3,
    'in_rack_stretch': 53}}},
 'compute_nodes_25': {'position': 25,
  'id': '25',
  'LAN_2__Z_9xxx_41': {'cable_count': 1,
   'speed': 400,
   'to_external_rack': 0,
   'full_no_split': 1,
   'cables_count': 1.0,
   'cable_type': {'model': '3m_400_copper_eth',
    'server_port_speed': 400,
    'split': 1,
    'length': 3,
    'in_rack_stretch': 53}},
  'LAN_3__S_xx64_39': {'cable_count': 1,
   'speed': 25,
   'to_external_rack': 0,
   'full